# 03 · RAG with OCR: what OCR recovers, and where it fails

**Filing:** Chevron FY2022. 52 of its 116 pages have a **garbled text layer** (broken font encoding), including the financial statements on pp. 58–62. Page 35 has **production charts that exist only as pixels**.

**Steps:**
1. Show that the native text of the garbled pages is unusable.
2. **Full-page OCR** of the garbled statement pages (`PageOCR`, RapidOCR). Check the result against the **exact XBRL figures**.
3. **Image-region OCR** of the chart page (`SelectiveOCR`), in two passes at different render scales, checked against the chart gold values.
4. **QA before and after OCR:** the same questions over three indexes:
   - no OCR;
   - OCR front-first with a page limit (the ingest default);
   - OCR on every garbled page.

**Runtime:** GPU recommended. CPU OCR takes ~4–18 s per page; for GPU OCR, install `onnxruntime-gpu` and set `NB_OCR_CUDA=1`.

In [ ]:
# 0. Setup. On Colab: unpacks the code zip from Drive (else clones the repo) and installs packages. Locally: run from inside the repo.
import json, os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/rama159065/Capstone.git"
BRANCH = "finance_report_qa_agent"
CODE_ZIP = "/content/drive/MyDrive/capstone_rag/capstone_code.zip"   # python -m scripts.common.colab_bundle, then upload to Drive
IN_COLAB = "google.colab" in sys.modules


def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


def fetch_code(root, repo_url, branch):
    """Unpack the code zip (Drive, else one you upload) into root; without a zip, clone the repo."""
    import zipfile
    from google.colab import drive
    drive.mount("/content/drive")
    uploaded = Path("/content") / Path(CODE_ZIP).name
    source = next((p for p in (Path(CODE_ZIP), uploaded) if p.exists()), None)
    if source is None and not root.exists():
        from google.colab import files
        print(f"{CODE_ZIP} not found: upload capstone_code.zip (cancel to clone {repo_url} instead)")
        names = list(files.upload())
        source = Path("/content") / names[0] if names else None
    if source is not None:
        with zipfile.ZipFile(source) as archive:
            info = archive.read("BUNDLE_INFO.json")
            stamp = root / "BUNDLE_INFO.json"
            if not stamp.exists() or stamp.read_bytes() != info:      # a new build: unpack over the old one
                archive.extractall(root)
        print("code:", source, json.loads(info))
    elif not root.exists():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")      # needed while the repo is private
        except Exception:
            token = None
        url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url, str(root)], check=True)


if IN_COLAB:
    ROOT = Path("/content/Capstone")
    fetch_code(ROOT, REPO_URL, BRANCH)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qdrant-client>=1.12,<2", "pymupdf>=1.24,<2",
                    "pydantic>=2", "pandas", "rapidocr>=3,<4", "onnxruntime", "sentence-transformers"], check=True)
    for key in ("LLM_API_BASE", "LLM_API_KEY", "LLM_MODEL", "SEC_USER_AGENT"):
        if not os.getenv(key) and colab_secret(key):
            os.environ[key] = colab_secret(key)
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("SEC_USER_AGENT", "capstone-research your.name@example.com")   # SEC requires your real e-mail
print("repo:", ROOT, "| Colab:", IN_COLAB)

**LLM.** `NB_LLM=auto` picks, in order:
1. **The provided Track B endpoint**, when `LLM_API_BASE` and `LLM_MODEL` are set (Colab secrets or environment). Grade with this one.
2. **A small Hugging Face stand-in** (`Qwen/Qwen2.5-1.5B-Instruct`), on a Colab GPU.
3. **The deterministic mock** otherwise. It only proves that the pipeline runs.

Force a choice with `os.environ["NB_LLM"] = "hosted" | "transformers" | "mock"` before the config cell.

In [ ]:
import re
import pymupdf
from scripts.common import notebook_helpers as nb
from scripts.rag import financial_ingest_qdrant_db as fi
from scripts.rag.financial_ingest_qdrant_db import PageOCR
from scripts.rag.selective_ocr import SelectiveOCR

DOC = "CVX_2022"
WORK = Path(os.getenv("NB_WORK", "/content/nb03_ocr" if IN_COLAB else "tmp/nb03_ocr"))
OCR_CUDA = os.getenv("NB_OCR_CUDA") == "1"
MAX_STATEMENT_PAGES = int(os.getenv("NB_OCR_PAGES", "12"))       # garbled table pages to OCR in step 2
EMBEDDING = os.getenv("NB_EMBED", "local")
EMBED_MODEL = os.getenv("NB_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
llm, llm_name = nb.make_llm(os.getenv("NB_LLM", "auto"))
row = nb.pick_filings([DOC])[0]
pdf = nb.fetch_pdfs([row])[row["document_id"]]
pages = fi.analyse_pdf(pdf)
garbled = [p for p in pages if p["garbled"]]
print("LLM:", llm_name, f"| {len(pages)} pages, {len(garbled)} garbled:", [p["page"] for p in garbled])

## 1. The garbled text layer

In [ ]:
sample = next(p for p in garbled if p["page_type"] == "table")
print(f"page {sample['page']} native text:\n", sample["text"][:400])

## 2. Full-page OCR of the statement pages, checked against XBRL

In [ ]:
METRICS = ["revenue", "net income", "diluted EPS", "total assets", "cash and cash equivalents", "operating cash flow"]
gold = [g for g in (nb.xbrl_item(row["ticker"], m, row["year"], "Chevron") for m in METRICS) if g]


def printed_forms(item):
    value = item["gold_value"]["value"]
    return [f"{value:.2f}"] if item["gold_value"]["unit"] == "USD/shares" else [f"{value / 1e6:,.0f}", f"{value / 1e6:.0f}"]


ocr = PageOCR(cuda=OCR_CUDA)
targets = [p for p in garbled if p["page_type"] == "table"][:MAX_STATEMENT_PAGES]
ocr_text, seconds = {}, []
with pymupdf.open(pdf) as document:
    for p in targets:
        started = time.time()
        ocr_text[p["page"]] = ocr.read(document[p["page"] - 1])
        seconds.append(time.time() - started)
print(f"OCR'd {len(targets)} pages, {sum(seconds) / max(1, len(seconds)):.1f} s/page")
check = []
for item in gold:
    forms = printed_forms(item)
    native = [p["page"] for p in pages if any(f in p["text"] for f in forms)]
    found = [n for n, t in ocr_text.items() if any(f in t for f in forms)]
    check.append({"figure": item["question"].split("'s ")[1].split(" for")[0], "gold (as printed)": forms[0],
                  "in native text (pages)": native, "in OCR text (pages)": found})
nb.show(nb.table(check))

## 3. Image-region OCR on the chart page, two passes

In [ ]:
CHART_PAGE = 35
chart_items = [i for i in nb.eval_items([DOC], categories=("chart_value",), general=False)]
passes = {}
for scale in ("2", "3"):
    os.environ["RAG_OCR_SCALE"] = scale
    reader = SelectiveOCR()
    with pymupdf.open(pdf) as document:
        result = reader.extract(document[CHART_PAGE - 1])
    passes[scale] = result
    print(f"scale {scale}: mode={result.mode}, regions={len(result.regions)}, lines={len(result.blocks)}, "
          f"rejected low-confidence lines={result.rejected_lines}")
os.environ.pop("RAG_OCR_SCALE")
numbers = {s: set(re.findall(r"\d[\d,.]*\d|\d", r.text)) for s, r in passes.items()}
print("numbers read in both passes:", sorted(numbers["2"] & numbers["3"]))
print("disagreements (only one pass):", sorted(numbers["2"] ^ numbers["3"]))
for item in chart_items:
    wanted = item["gold_value"]["values"] if item.get("gold_value") else []
    print(f"{item['id']}: gold {wanted} -> found in OCR: {[v for v in wanted if v.replace(',', '') in passes['3'].text.replace(',', '')]}")

## 4. QA before and after OCR

In [ ]:
SCENARIOS = [s for s in os.getenv("NB_OCR_SCENARIOS", "no_ocr,ocr_front_first_20,ocr_all").split(",") if s]
settings = {"no_ocr": ("none", 0), "ocr_front_first_20": ("garbled", 20), "ocr_all": ("garbled", 0)}
settings.update({s: ("garbled", int(s.rsplit("_", 1)[1])) for s in SCENARIOS if s.startswith("ocr_front_first_")})
manifest = WORK / "manifest.jsonl"
WORK.mkdir(parents=True, exist_ok=True)
manifest.write_text(json.dumps(row) + "\n", encoding="utf-8")
indexes = {}
for scenario in SCENARIOS:
    mode, limit = settings[scenario]
    argv = ["ingest", "--root", str(WORK / scenario), "--manifest", str(manifest), "--pdf-dir", str(nb.PDF_DIR),
            "--stages", "text", "--ocr", mode, "--ocr-max-pages-per-filing", str(limit), "--embedding-provider", EMBEDDING,
            "--embedding-model", EMBED_MODEL, "--workers", "1", "--reset"] + (["--ocr-cuda"] if OCR_CUDA else [])
    started = time.time()
    ingest = fi.Ingest(fi.build_parser().parse_args(argv))
    ingest.run()
    cache = json.loads((WORK / scenario / "pages" / f"{row['document_id']}.json").read_text(encoding="utf-8"))["pages"]
    indexes[scenario] = ingest
    print(f"{scenario}: {time.time() - started:.0f}s, OCR'd pages {sum(bool(p.get('ocr_signature')) for p in cache)}, "
          f"points {ingest.client.count(fi.TEXT, exact=True).count}")

questions = gold + chart_items
rows_out = []
for scenario, ingest in indexes.items():
    fi._serving.clear()
    fi._serving.update(client=ingest.client, embedder=ingest.embedder)
    for item in questions:
        hits = nb.label_hits(fi.retrieve_for_agent(item["question"], top_k=5, ticker=row["ticker"], year=row["year"]))
        reply = nb.run_template(llm, "chain_of_thought", item["question"], hits)
        result = nb.score(item, reply, hits)
        rows_out.append({"scenario": scenario, "id": item["id"], "category": item["category"], "correct": result["correct"],
                         "gold_value_in_evidence": result["gold_value_in_evidence"], "failure": result["failure"],
                         "pages": [h["page"] for h in hits]})
qa = nb.table(rows_out)
nb.show(qa.pivot_table(index="id", columns="scenario", values="gold_value_in_evidence", aggfunc="first"))
nb.show(qa.groupby("scenario")[["gold_value_in_evidence", "correct"]].mean().round(2))

## OCR failure scenarios

**Measured above:**
- **Not OCR'd at all.** With the front-first page limit, the statements on pp. 58–62 are never OCR'd, so their figures stay unretrievable (`ocr_front_first_20` against `ocr_all`).
- **Missing or altered digits.** A figure is absent from the OCR text, or appears with a changed digit or separator (`in OCR text` is empty).
- **Low-confidence lines dropped.** The confidence threshold is 0.6 (`rejected_lines`). Small chart labels are the usual loss.
- **Pass disagreement.** The two render scales read different numbers. Design tier T4 accepts a chart value only when both passes agree **and** the geometry checks pass.

**Known from the corpus study:**
- **False numbers.** Both passes can read an axis tick as a value: Chevron's "12000" is really the "2,000" tick. Agreement between passes alone is not proof.
- **Lost layout.** OCR returns lines, not table cells, so column alignment (2022 vs 2021) and the "in millions" scale can be lost. Expect row and column mix-ups in answers.
- **Unlabelled chart segments.** No printed value exists, so OCR has nothing to read. The answer must say the value isn't printed.
- **Cost.** ~4–18 s/page on CPU. Use GPU OCR, OCR only unreadable pages, and take exact figures from the 10-K XBRL wherever possible.
- **Chart numbers on clean pages are never indexed.** At ingest, OCR runs only on garbled or outlined pages. Page 35 has a clean text layer, with its chart numbers only in pixels, so step 3 can read them but the index never stores them. Retrieved evidence holds 0 chart values in every scenario. The fix (still to build) is image-region OCR at ingest, saving validated values as `chart_value` facts.

**Measured on a local CPU run (Chevron FY2022):**

| Check | Result |
|---|---|
| XBRL statement figures in the native text | 0 of 6 |
| Full-page OCR of 12 statement pages | 5 of 6 found, at 4.5 s/page. Operating cash flow was on a page outside those 12 |
| Chart page, two passes | all 4 gold values read (1,440; 7,677; 11.2; 279), plus the false "12,000" in both passes |
| Gold value in the retrieved evidence | no OCR 20% → front-first 20 pages 40% → all 52 garbled pages 50% |